# 証券営業インテリジェンス ハンズオン
## Part 2: Cortex AI Functions

このノートブックでは、Snowflake Cortex AI Functions を使って非構造化データ（ニュース・アナリストレポート等）を自動的に処理します。

### Cortex AI Functions とは

SQLの中でAI処理を直接呼び出せる関数群です。機械学習の知識は不要で、SQLを書けるだけで使えます。

| 関数 | 説明 | 本ハンズオンでの用途 |
|---|---|---|
| **AI_SUMMARIZE** / **AI_COMPLETE** | テキスト要約・質問応答 | アナリストレポートの要約 |
| **AI_SENTIMENT** | 感情分析（ポジティブ/ネガティブ/ニュートラル）| ニュースの感情分析 |
| **AI_EXTRACT** | 構造化情報の抽出 | ニュースからの銘柄・イベント情報抽出 |
| **AI_CLASSIFY** | テキストの分類 | ニュースカテゴリ分類 |

### 🚀 アハ体験ポイント

> **「1,000文字のアナリストレポートが3秒で3行に要約される」**
> 50本のニュース記事を一括でセンチメント分析 → 保有銘柄への影響を即座に把握

In [ ]:
%%sql -r result_use
USE DATABASE SNOWFINANCE_DB;
USE SCHEMA DEMO_SCHEMA;
USE WAREHOUSE DEMO_WH;
SELECT 'Ready!' AS STATUS;

## Step 1: AI_SUMMARIZE / AI_COMPLETE - テキスト要約

### 1-1. アナリストレポートの瞬時要約

1,000文字超のアナリストレポートを**3行に要約**します。
RM（リレーションシップマネージャー）が顧客訪問前に5分で情報収集できます。

In [ ]:
%%sql -r result_summary
-- ============================================================
-- AI_COMPLETE でアナリストレポートを要約（3行以内）
-- ============================================================
SELECT
    SECURITY_NAME       AS 銘柄,
    RATING              AS 投資判断,
    TARGET_PRICE        AS 目標株価,
    PUBLISH_DATE        AS 発行日,
    SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large2',
        CONCAT(
            '以下のアナリストレポートを証券営業担当者向けに3行以内で要約してください。',
            '【投資判断の変化】【投資ポイント】【主なリスク】の観点で簡潔にまとめてください。\n\n',
            'レポートタイトル: ', REPORT_TITLE, '\n',
            '要約: ', EXECUTIVE_SUMMARY, '\n',
            '投資根拠: ', INVESTMENT_THESIS, '\n',
            'リスク: ', KEY_RISKS
        )
    ) AS AI要約
FROM ANALYST_REPORTS
WHERE SECURITY_CODE = '7203'  -- トヨタ自動車
ORDER BY PUBLISH_DATE DESC
LIMIT 1;

### 1-2. 顧客訪問前ブリーフィング自動生成

顧客IDを指定するだけで、訪問前に必要な情報を**まとめて1枚のブリーフィングシート**として生成します。

> **アハ体験**: 準備に1時間かかっていた作業が**30秒**で完了！

In [ ]:
%%sql -r result_briefing
-- 顧客C001（山田太郎様）の訪問前ブリーフィング自動生成
SELECT
    SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large2',
        CONCAT(
            '以下の顧客情報と保有銘柄情報をもとに、証券営業担当者向けの訪問前ブリーフィングを作成してください。',
            '【顧客サマリー】【主な保有銘柄と含み益/損】【今日の提案機会】の3セクションで。\n\n',
            '顧客名: ', c.CUSTOMER_NAME,
            '  年齢: ', c.AGE, '歳',
            '  セグメント: ', c.SEGMENT,
            '  総資産: ', TO_VARCHAR(c.TOTAL_ASSETS/100000000, '99.9'), '億円',
            '  投資目的: ', c.INVESTMENT_PURPOSE,
            '  リスク許容度: ', c.RISK_TOLERANCE, '\n',
            '  メモ: ', COALESCE(c.NOTES, '特になし'), '\n',
            '  ライフイベント: ', COALESCE(e.EVENT_TYPE || '（' || TO_VARCHAR(e.ESTIMATED_AMOUNT/10000000,'999') || '千万円必要）', '特になし'), '\n',
            '  主な保有銘柄: トヨタ自動車（時価2,850万円・含み益1,350万円）、ソニー（時価7,600万円）、KDDI（時価3,720万円）'
        )
    ) AS 訪問前ブリーフィング
FROM DIM_CUSTOMER c
LEFT JOIN DIM_LIFE_EVENT e ON c.CUSTOMER_ID = e.CUSTOMER_ID AND e.STATUS = '検討中'
WHERE c.CUSTOMER_ID = 'C001'
LIMIT 1;

## Step 2: AI_SENTIMENT - 感情分析

### 2-1. マーケットニュースの感情分析

50件のニュース記事を**一括で感情分析**します。
ポジティブ・ネガティブ・ニュートラルを自動判定。

> **アハ体験**: 毎朝50本の記事を読んでいたRMが、今日から**ダッシュボード1画面**で把握できる！

In [ ]:
%%sql -r result_sentiment
-- ニュース記事の感情分析（上位10件）
SELECT
    NEWS_ID,
    PUBLISH_DATE,
    CATEGORY,
    LEFT(TITLE, 50) AS タイトル（先頭50字）,
    SENTIMENT AS 登録済みセンチメント,
    SNOWFLAKE.CORTEX.AI_SENTIMENT(CONTENT):categories[0].name::VARCHAR AS AI感情分析結果,
    SNOWFLAKE.CORTEX.AI_SENTIMENT(CONTENT):categories[0].sentiment::FLOAT AS 感情スコア
FROM NEWS_ARTICLES
WHERE RELATED_SECURITIES LIKE '%7203%' OR RELATED_SECURITIES LIKE '%トヨタ%'
ORDER BY PUBLISH_DATE DESC;

In [ ]:
%%sql -r result_sentiment_all
-- 全ニュースの感情分析サマリー（保有銘柄への影響を一目で確認）
SELECT
    SENTIMENT               AS 事前分類,
    COUNT(*)                AS 件数,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS 割合_PCT
FROM NEWS_ARTICLES
GROUP BY SENTIMENT
ORDER BY 件数 DESC;

## Step 3: AI_EXTRACT - 構造化情報の抽出

### 3-1. ニュースから銘柄・カテゴリを自動抽出

非構造化テキストから**JSON形式**で情報を抽出します。
「トヨタ」「税制改正」「相続」などのキーワードを自動検出。

In [ ]:
%%sql -r result_extract
-- ニュースから重要情報を構造化抽出
SELECT
    NEWS_ID,
    PUBLISH_DATE,
    LEFT(TITLE, 60) AS タイトル,
    SNOWFLAKE.CORTEX.AI_EXTRACT(
        CONTENT,
        OBJECT_CONSTRUCT(
            'related_stocks',     '関連する株式銘柄コードまたは企業名（カンマ区切り）',
            'event_type',         'イベント種別（決算/税制改正/M&A/格付変更/経済指標/その他）',
            'impact_direction',   '株価や市場への影響方向（上昇要因/下落要因/ニュートラル）',
            'urgency',            '緊急度（高/中/低）'
        )
    ) AS 抽出情報
FROM NEWS_ARTICLES
WHERE IMPORTANCE = '高'
ORDER BY PUBLISH_DATE DESC
LIMIT 5;

## Step 4: AI_CLASSIFY - テキスト分類

### 4-1. ニュースを営業活動タイプ別に分類

「この記事は顧客への提案材料になるか？」を自動分類します。

In [ ]:
%%sql -r result_classify
-- ニュースを営業活動タイプで分類
SELECT
    NEWS_ID,
    PUBLISH_DATE,
    LEFT(TITLE, 55) AS タイトル,
    SNOWFLAKE.CORTEX.AI_CLASSIFY(
        CONTENT,
        ['株式売買タイミング提案',
         '相続・贈与対策提案',
         '新商品・サービス紹介',
         'マーケット情報提供',
         '税制・規制変更対応']
    ):labels[0]::VARCHAR AS 営業活動分類,
    SNOWFLAKE.CORTEX.AI_CLASSIFY(
        CONTENT,
        ['株式売買タイミング提案',
         '相続・贈与対策提案',
         '新商品・サービス紹介',
         'マーケット情報提供',
         '税制・規制変更対応']
    ):labels[0]:score::FLOAT AS 確信度
FROM NEWS_ARTICLES
ORDER BY PUBLISH_DATE DESC
LIMIT 10;

## Step 5: 総合活用 - 山田様への提案材料を自動生成

上記のAI Functionsを組み合わせて、山田様（C001）への提案に関連するニュースを抽出・要約・分類します。

> **アハ体験**: 山田様の保有銘柄（トヨタ）に関連するニュースを自動的に抽出し、提案材料として整理！

In [ ]:
%%sql -r result_integrated
-- 山田様関連ニュースの一括AI処理
WITH yamada_securities AS (
    SELECT DISTINCT SECURITY_CODE
    FROM FACT_PORTFOLIO
    WHERE CUSTOMER_ID = 'C001'
),
relevant_news AS (
    SELECT n.*
    FROM NEWS_ARTICLES n
    WHERE EXISTS (
        SELECT 1 FROM yamada_securities s
        WHERE n.RELATED_SECURITIES LIKE '%' || s.SECURITY_CODE || '%'
    )
    OR n.TAGS LIKE '%相続%' OR n.TAGS LIKE '%贈与%' OR n.TAGS LIKE '%税制%'
)
SELECT
    PUBLISH_DATE                AS 日付,
    LEFT(TITLE, 50)             AS タイトル,
    SENTIMENT                   AS センチメント,
    SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large2',
        CONCAT('次のニュースを30文字以内で山田様への提案観点から一言でまとめてください: ', TITLE, ' ', SUMMARY)
    ) AS 提案ポイント（30字）
FROM relevant_news
ORDER BY IMPORTANCE DESC, PUBLISH_DATE DESC
LIMIT 8;

## まとめ

Part 2 完了！Cortex AI Functionsを体験しました。

### 使用した AI Functions

| 関数 | 実行内容 |
|---|---|
| `AI_COMPLETE` | アナリストレポート要約・訪問前ブリーフィング生成 |
| `AI_SENTIMENT` | ニュース感情分析（ポジティブ/ネガティブ）|
| `AI_EXTRACT` | ニュースから銘柄・イベント種別・影響方向を抽出 |
| `AI_CLASSIFY` | ニュースを営業活動タイプで分類 |

### ポイント
- **SQLのみで実装**: Pythonや機械学習の知識は不要
- **マルチモデル対応**: `mistral-large2`, `llama3.1-70b`, `claude-3-5-sonnet` など選択可
- **本番データに直接適用可能**: 新しいテーブルを作らずにSQLで加工

**次のステップ:** `03_cortex_analyst.ipynb` で自然言語によるデータ分析を体験してください。